In [4]:
import os
import glob
import pandas as pd
import nibabel as nib
import numpy as np

def calculate_volume_area(mask_path):
    """Calculate volume and max area from mask"""
    mask = nib.load(mask_path)
    mask_data = mask.get_fdata()
    voxel_spacing = mask.header.get_zooms()
    
    # Volume (rounded to 5 decimals)
    voxel_volume = voxel_spacing[0] * voxel_spacing[1] * voxel_spacing[2]
    volume = round(np.sum(mask_data > 0) * voxel_volume, 5)
    
    # Max area (rounded to 5 decimals)
    pixel_area = voxel_spacing[0] * voxel_spacing[1]
    pixels_per_slice = np.sum(mask_data > 0, axis=(0, 1))
    max_area = round(np.max(pixels_per_slice) * pixel_area, 5)
    
    return volume, max_area

# Config
CSV_FILE = 'proceed_radiomics_103_patients(two_years).csv'
MASK_FOLDER = './convert_nifti_format'
OUTPUT_FILE = 'cmc_complete.csv'

# Load CSV
df = pd.read_csv(CSV_FILE)
print(f"Processing {len(df)} patients...")

volumes = []
areas = []
missing = 0

for idx, row in df.iterrows():
    patient_id = row['id']
    
    # Find mask file
    mask_pattern = f"{MASK_FOLDER}/{patient_id}/mask_*_re.nii.gz"
    mask_files = glob.glob(mask_pattern)
    
    if not mask_files:
        print(f"⚠ Missing: {patient_id}")
        volumes.append(np.nan)
        areas.append(np.nan)
        missing += 1
        continue
    
    try:
        vol, area = calculate_volume_area(mask_files[0])
        volumes.append(vol)
        areas.append(area)
        if (idx + 1) % 20 == 0:
            print(f"Processed {idx + 1}/{len(df)}...")
    except Exception as e:
        print(f"❌ Error {patient_id}: {e}")
        volumes.append(np.nan)
        areas.append(np.nan)
        missing += 1

# Update and save
df['vol'] = volumes
df['area'] = areas
df.to_csv(OUTPUT_FILE, index=False)

print(f"\n✓ Saved: {OUTPUT_FILE}")
print(f"  Success: {len(df) - missing}/{len(df)}")
print(f"  Volume: {df['vol'].mean():.2f} mm³ (mean)")
print(f"  Area: {df['area'].mean():.2f} mm² (mean)")

Processing 102 patients...
Processed 20/102...
Processed 40/102...
Processed 60/102...
Processed 80/102...
Processed 100/102...

✓ Saved: cmc_complete.csv
  Success: 102/102
  Volume: 13311.36 mm³ (mean)
  Area: 516.06 mm² (mean)


---